# Sistemas Inteligentes – Exercícios 7, 8 e 9

- **Exercício 7**: `KNNRegressor` e RMSE.
- **Exercício 8**: RidgeRegressionLeastSquares (regressão).
- **Exercício 9**: RandomForestClassifier (classificação, iris).

In [1]:
from pathlib import Path
import numpy as np

from si.io.csv_file import read_csv
from si.data.dataset import Dataset
from si.model_selection.split import train_test_split, stratified_train_test_split
from si.models.ridge_regression_least_squares import RidgeRegressionLeastSquares
from si.metrics.mse import mse
from si.metrics.rmse import rmse
from si.models.knn_regressor import KNNRegressor
from si.models.random_forest_classifier import RandomForestClassifier
from si.ensemble.stacking_classifier import StackingClassifier
from si.models.knn_classifier import KNNClassifier
from si.models.logistic_regression import LogisticRegression
from si.models.decision_tree_classifier import DecisionTreeClassifier

iris_path = Path(r"C:\Users\filip\OneDrive\Attachments\Ambiente de Trabalho\SIB_gh\si2\datasets\iris\iris.csv")
iris_ds = read_csv(iris_path, sep=",", features=True, label=True)
X, y = iris_ds.X, iris_ds.y

cpu_path = Path(r"C:\Users\filip\OneDrive\Attachments\Ambiente de Trabalho\SIB_gh\si2\datasets\cpu\cpu.csv")
cpu_ds = read_csv(cpu_path, sep=",", features=True, label=True)

print("Shape do dataset cpu:", cpu_ds.X.shape)
print("Primeiras features:", cpu_ds.features[:5])

Shape do dataset cpu: (209, 6)
Primeiras features: Index(['syct', 'mmin', 'mmax', 'cach', 'chmin'], dtype='object')


# Exercício 7

**Exercício 7 – Exemplo de regressão: y = 2x + 1**

In [2]:
train_cpu, test_cpu = train_test_split(cpu_ds, test_size=0.3, random_state=42)

knn_reg = KNNRegressor(k=5)
knn_reg.fit(train_cpu)

rmse_train = knn_reg.score(train_cpu)
rmse_test = knn_reg.score(test_cpu)

print("RMSE Train:", rmse_train)
print("RMSE Test:", rmse_test)

RMSE Train: 44.88600224211231
RMSE Test: 151.41521634873914


In [3]:
rng = np.random.default_rng(42)
X_reg = rng.uniform(-5, 5, size=(100, 1))
y_reg = 2 * X_reg[:, 0] + 1

reg_ds = Dataset(X=X_reg, y=y_reg, features=["x"], label="y")

# split manual train/test
indices = rng.permutation(X_reg.shape[0])
train_size = int(0.7 * X_reg.shape[0])
train_idx, test_idx = indices[:train_size], indices[train_size:]

train_ds_reg = Dataset(X=X_reg[train_idx], y=y_reg[train_idx], features=["x"], label="y")
test_ds_reg = Dataset(X=X_reg[test_idx], y=y_reg[test_idx], features=["x"], label="y")

model_knn = KNNRegressor(k=3)
model_knn.fit(train_ds_reg)

y_pred_train = model_knn.predict(train_ds_reg)
y_pred_test = model_knn.predict(test_ds_reg)

print("RMSE Train:", rmse(train_ds_reg.y, y_pred_train))
print("RMSE Test:", rmse(test_ds_reg.y, y_pred_test))

RMSE Train: 0.2036120166901317
RMSE Test: 0.38259592850513513


# Exercicio 8

**Exercício 8 – Ridge Regression (dataset de regressão, por ex. cpu.csv)**

In [4]:
train_cpu, test_cpu = train_test_split(cpu_ds, test_size=0.3, random_state=42)

ridge = RidgeRegressionLeastSquares(l2_penalty=1.0, scale=True)
ridge.fit(train_cpu)

y_pred_train = ridge.predict(train_cpu)
y_pred_test = ridge.predict(test_cpu)

print("MSE Train:", mse(train_cpu.y, y_pred_train))
print("MSE Test:", mse(test_cpu.y, y_pred_test))

MSE Train: 1438.6687527909294
MSE Test: 15007.239494047795


# Exercicio 9

**Exercício 9 – RandomForestClassifier + iris**

In [5]:
train_iris, test_iris = stratified_train_test_split(iris_ds, test_size=0.3, random_state=42)

rf = RandomForestClassifier(
    n_estimators=50,
    max_features=2,
    min_sample_split=2,
    max_depth=5,
    mode="gini",
    seed=42
)

rf.fit(train_iris)

print("Accuracy Train:", rf.score(train_iris))
print("Accuracy Test:", rf.score(test_iris))

Accuracy Train: 1.0
Accuracy Test: 0.9333333333333333
